**参考答案版**。请先独立完成 `lessons/` 里的练习,再来对照。

# Day 01 · 信度与效度：先检验测量工具

**心理大模型评测与 Agent** · 评测方法学 · 建议用时 3 到 4 小时

**今天的目标**

- 用经典测量理论解释信度，并说明信度是效度的必要不充分条件
- 区分重测信度、内部一致性和评分者间信度
- 把这些概念对应到两类问题：LLM 当裁判，以及让 LLM 填心理量表
- 用模拟数据展示一个稳定但不准确的裁判

**怎么用这个 notebook**：先看同一文件夹里的 `slides.html`，再从上到下运行这里的单元格。标着 **练习** 的单元格需要你自己填写，填完运行下面的批改单元格，会显示 `正确`、`再想想` 或 `待完成`。所有数据都是课程编写或模拟的，没有真实来访者数据。

In [ ]:
# 课程环境设置:让 notebook 找到课程代码库 src/course
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "course").exists())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from course.plotting import use_style, OCHRE, TEAL, ROSE, SAND, INK
from course.check import check, check_true, summary
use_style()
pd.set_option("display.precision", 3)

## 1. 经典测量理论：用模拟来理解

我们模拟 1000 位来访者的抑郁真分 T，再测量两次，每次加入独立误差。两次测量的相关，就是信度的估计。

In [ ]:
rng = np.random.default_rng(0)
n = 1000
T = rng.normal(10, 4, n)          # 真分,标准差 4
E1 = rng.normal(0, 2, n)          # 第一次测量误差,标准差 2
E2 = rng.normal(0, 2, n)          # 第二次测量误差
X1, X2 = T + E1, T + E2

theoretical = 4**2 / (4**2 + 2**2)
print(f"理论信度 Var(T)/Var(X) = {theoretical:.3f}")
print(f"两次测量的相关         = {np.corrcoef(X1, X2)[0,1]:.3f}")

**练习 1.1**：如果误差标准差从 2 增加到 4（例如来访者在嘈杂环境中用手机作答），理论信度是多少？把答案填到 `ans_1_1`。

In [ ]:
ans_1_1 = 16 / (16 + 16)
check("练习 1.1 误差 SD=4 时的理论信度", ans_1_1, 16 / (16 + 16), tol=0.01)

## 2. 心理量表的信度：PHQ-9 风格的模拟数据

`simulate_phq9` 生成 9 个题目（0 到 3 分），由同一个潜在抑郁严重度驱动，并在两周后重测。下面计算内部一致性（Cronbach's α）。

In [ ]:
from course.data import simulate_phq9
from course.audit import cronbach_alpha

t1, t2, latent = simulate_phq9(n=800, retest_r=0.85, seed=1)
print("第一次测量的 α =", round(cronbach_alpha(t1), 3))
print("总分范围:", t1.sum(1).min(), "到", t1.sum(1).max())

**练习 1.2**：计算两次测量总分的重测相关（Pearson r)，填到 `ans_1_2`。提示：`t1.sum(1)` 是每个人的总分。

In [ ]:
ans_1_2 = np.corrcoef(t1.sum(1), t2.sum(1))[0, 1]
check("练习 1.2 PHQ 总分的重测相关", ans_1_2, np.corrcoef(t1.sum(1), t2.sum(1))[0, 1], tol=0.005)

## 3. 稳定但不准确的裁判

模拟 120 条聊天机器人回复，每条有真实质量（1 到 5）和长度。
- 裁判 A：围绕真实质量随机波动，噪声大，没有系统偏差。
- 裁判 B：噪声小，但主要受回复长度影响。

每个裁判对同一批回复打分两次。

In [ ]:
rng = np.random.default_rng(3)
q = rng.uniform(1.5, 4.5, 120)                 # 真实质量
words = rng.normal(120, 50, 120).clip(20, 300) # 回复字数
def judge_A(): return q + rng.normal(0, 0.8, 120)
def judge_B(): return 1.2 + 0.25 * q + 0.012 * words + rng.normal(0, 0.15, 120)
A1, A2, B1, B2 = judge_A(), judge_A(), judge_B(), judge_B()

table = pd.DataFrame({
    "重测信度 r(run1, run2)": [np.corrcoef(A1, A2)[0,1], np.corrcoef(B1, B2)[0,1]],
    "与真实质量的相关": [np.corrcoef(A1, q)[0,1], np.corrcoef(B1, q)[0,1]],
    "与长度的相关": [np.corrcoef(A1, words)[0,1], np.corrcoef(B1, words)[0,1]],
}, index=["裁判 A", "裁判 B"])
table

**练习 1.3**：哪个裁判的重测信度更高？哪个裁判的效度（与真实质量的相关）更高？分别填 `"A"` 或 `"B"`。

In [ ]:
ans_1_3_reliability = "B"
ans_1_3_validity = "A"
check("练习 1.3a 重测信度更高的裁判", ans_1_3_reliability, "B")
check("练习 1.3b 效度更高的裁判", ans_1_3_validity, "A")

## 4. 衰减校正：一致性的上限

裁判和咨询师都有测量误差。两者分数的相关，上限是 $\sqrt{r_{xx} \cdot r_{yy}}$（两者信度的几何平均）。Spearman 衰减校正公式估计“去掉误差后”的相关：

$$ r_{true} = \frac{r_{xy}}{\sqrt{r_{xx} \, r_{yy}}} $$

这告诉我们：如果咨询师之间本身就只有中等一致，就不能要求裁判与咨询师高度一致。

**练习 1.4**：裁判与咨询师评分相关 0.50，裁判重测信度 0.70，咨询师评分者间信度 0.80。校正后的相关是多少？

In [ ]:
ans_1_4 = 0.5 / np.sqrt(0.7 * 0.8)
check("练习 1.4 衰减校正后的相关", ans_1_4, 0.5 / np.sqrt(0.7 * 0.8), tol=0.01)

## 5. 写下来（不批改，但要写）

用三句话回答：为什么“裁判打分很稳定”不能说明“裁判打分很准”？你会用什么证据证明一个裁判有效？

*在这里写你的回答。*

## 口述题（录音，每题 90 秒）

1. 裁判打分很稳定，能说明它准吗？
   <details><summary>参考要点</summary>不能。稳定是信度，只排除随机误差。系统偏差（偏好长回复、偏好同源模型）可以同时很稳定。要用专家评分做效标，并做位置互换、长度控制等偏差检验。</details>
2. 信度和效度是什么关系？举一个 LLM 评测的例子。
   <details><summary>参考要点</summary>信度是效度的必要不充分条件。例子：一个按长度打分的裁判，重测一致性很高，但与咨询师的评分相关低。</details>

## 研究延伸

你在博士研究中做过量表测量（UTAUT2 问卷、RCT 结局量表）。可以把“咨询师加 rubric”当作一个新的评价工具，按量表开发的流程收集信效度证据：内容效度（专家评审）、评分者间信度、与外部标准的相关。这本身可以构成一项方法学研究。

In [ ]:
summary()